In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle

In [2]:
# load the trained model , scaler , onehot pickle
import os

project_dir = r"D:\ML projects\Customer Churn Prediction System"

model_path = os.path.join(project_dir, 'model.h5')
gender_path = os.path.join(project_dir, 'label_encoder_gender.pkl')
geo_path = os.path.join(project_dir, 'one_hot_encoder_geo.pkl')
scaler_path = os.path.join(project_dir, 'scaler.pkl')

model = load_model(model_path) 

# load scaler and encoder file
with open(gender_path , 'rb') as file:
    gender_encoder = pickle.load(file)

with open(geo_path , 'rb') as file:
    geo_encoder = pickle.load(file)

with open(scaler_path, 'rb') as file:
    scaler = pickle.load(file)


### Custom Input Data

In [3]:
# Example input data
input_data = {
'CreditScore': 600,
'Geography': 'France',
'Gender': 'Male',
'Age': 40,
'Tenure': 3,
'Balance': 60000,
'NumOfProducts': 2,
'HasCrCard': 1,
'IsActiveMember': 1,
'EstimatedSalary': 50000
}

In [4]:
# One-hot encode geography
geo_encoded = geo_encoder.transform([[input_data['Geography']]])
geo_encoded_df = pd.DataFrame(geo_encoded , columns=geo_encoder.get_feature_names_out(['Geography']))
geo_encoded_df


d:\ML projects\Customer Churn Prediction System\venvforchurn\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [6]:
# Encode the input gender
input_df['Gender'] = gender_encoder.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
# concatination one-hot encoded data
input_df = pd.concat([input_df.drop("Geography" , axis=1) , geo_encoded_df] , axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [8]:
# scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

d:\ML projects\Customer Churn Prediction System\venvforchurn\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


array([[6.00000000e+02, 1.00000000e+00, 4.00000000e+01, 3.00000000e+00,
        6.00000000e+04, 2.00000000e+00, 1.00000000e+00, 1.00000000e+00,
        5.00000000e+04, 1.00000000e+00, 4.26325641e-17, 7.19424520e-17]])

In [9]:
# predict churn
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step


array([[0.]], dtype=float32)

In [10]:
prediction_prob = prediction[0][0]
prediction_prob

np.float32(0.0)

In [11]:
if prediction_prob > 0.5:
    print("The customer is likely to churn")
else:
    print("The customer is not likely to churn")

The customer is not likely to churn
